# F1 Analytics — Exploración y validación de modelos ML

Ejecuta las celdas en orden. Ajusta los paths a tus CSVs antes de empezar.

In [ ]:
import sys
sys.path.insert(0, '..')  # para importar desde scripts/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── AJUSTA ESTOS PATHS ──────────────────────────────────────────
DATA_DIR    = '/ruta/a/ergast/csvs/'       # carpeta con races.csv, results.csv, etc.
EVENTS_CSV  = '/ruta/a/race_events.csv'    # dataset jtrotman
# ────────────────────────────────────────────────────────────────

print('Paths configurados')

## 1. Explorar Race Events

In [ ]:
events = pd.read_csv(EVENTS_CSV, low_memory=False)
print(f'Shape: {events.shape}')
print(f'Columnas: {list(events.columns)}')
events.head(3)

In [ ]:
# Ver todos los tipos de evento únicos
col = 'event_type'  # ← cambia si el nombre difiere
print(events[col].value_counts().to_string())

## 2. Dataset Safety Car

In [ ]:
from scripts.feature_engineering import build_safety_car_features

df_sc = build_safety_car_features(
    races_csv       = DATA_DIR + 'races.csv',
    circuits_csv    = DATA_DIR + 'circuits.csv',
    results_csv     = DATA_DIR + 'results.csv',
    race_events_csv = EVENTS_CSV,
    pit_stops_csv   = DATA_DIR + 'pit_stops.csv',
)

print(df_sc.shape)
df_sc.describe()

In [ ]:
# Evolución de tasa de SC por año
sc_by_year = df_sc.groupby('year')['had_safety_car'].mean()
sc_by_year.plot(kind='bar', figsize=(14, 4), title='Tasa de Safety Car por temporada')
plt.ylabel('Proporción de carreras con SC')
plt.tight_layout()
plt.show()

## 3. Dataset Fallo Mecánico

In [ ]:
from scripts.feature_engineering import build_mechanical_failure_features

df_dnf = build_mechanical_failure_features(
    races_csv        = DATA_DIR + 'races.csv',
    results_csv      = DATA_DIR + 'results.csv',
    status_csv       = DATA_DIR + 'status.csv',
    drivers_csv      = DATA_DIR + 'drivers.csv',
    constructors_csv = DATA_DIR + 'constructors.csv',
    lap_times_csv    = DATA_DIR + 'lap_times.csv',
)

print(df_dnf.shape)
df_dnf['is_mechanical'].value_counts()

In [ ]:
# DNF mecánicos por constructor (top 15)
import pandas as pd

results = pd.read_csv(DATA_DIR + 'results.csv')
status  = pd.read_csv(DATA_DIR + 'status.csv')
constructors = pd.read_csv(DATA_DIR + 'constructors.csv')

from scripts.feature_engineering import MECHANICAL_DNF_STATUSES
status['is_mech'] = status['status'].isin(MECHANICAL_DNF_STATUSES).astype(int)

merged = results.merge(status[['statusId','is_mech']], on='statusId')
merged = merged.merge(constructors[['constructorId','name']], on='constructorId')
dnf_by_constructor = merged.groupby('name')['is_mech'].mean().sort_values(ascending=False).head(15)
dnf_by_constructor.plot(kind='barh', figsize=(8, 6), title='Tasa DNF mecánico por constructor (top 15)')
plt.tight_layout()
plt.show()

## 4. Entrenar y evaluar modelos

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, RocCurveDisplay, ConfusionMatrixDisplay

feature_cols_sc = [c for c in df_sc.columns if c not in ('raceId', 'had_safety_car')]
X_sc = df_sc[feature_cols_sc].values
y_sc = df_sc['had_safety_car'].values

pipe_sc = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('scl', StandardScaler()),
    ('clf', CalibratedClassifierCV(
        GradientBoostingClassifier(n_estimators=200, max_depth=4, random_state=42),
        cv=3, method='isotonic'
    ))
])

cv    = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
aucs  = cross_val_score(pipe_sc, X_sc, y_sc, cv=cv, scoring='roc_auc')
print(f'Safety Car CV AUC: {aucs.mean():.4f} ± {aucs.std():.4f}')

pipe_sc.fit(X_sc, y_sc)
probs = pipe_sc.predict_proba(X_sc)[:, 1]
print(f'Train AUC: {roc_auc_score(y_sc, probs):.4f}')

In [ ]:
# Curva ROC
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
RocCurveDisplay.from_predictions(y_sc, probs, ax=axes[0], name='Safety Car')
axes[0].set_title('Curva ROC — Safety Car')

# Distribución de probabilidades
axes[1].hist(probs[y_sc==0], bins=30, alpha=0.6, label='Sin SC')
axes[1].hist(probs[y_sc==1], bins=30, alpha=0.6, label='Con SC')
axes[1].set_title('Distribución de probabilidades')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Importancia de features (Safety Car)
import json

try:
    with open('../models_trained/training_metrics.json') as f:
        metrics = json.load(f)
    imp = metrics['safety_car']['feature_importance']
    pd.Series(imp).sort_values(ascending=True).plot(
        kind='barh', figsize=(8, 5), title='Feature importance — Safety Car'
    )
    plt.tight_layout()
    plt.show()
except FileNotFoundError:
    print('Ejecuta train_models.py primero para generar training_metrics.json')